In [ ]:
import torch
import psutil
import os
from sonar.inference_pipelines.text import TextToEmbeddingModelPipeline, EmbeddingToTextModelPipeline


In [ ]:
def get_memory_usage():
    """Get current memory usage in MB"""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 * 1024)

def count_parameters(model):
    """Count total and trainable parameters in a model"""
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total_params, trainable_params

def get_model_size_mb(model):
    """Estimate model size in MB"""
    param_size = 0
    buffer_size = 0
    
    for param in model.parameters():
        param_size += param.nelement() * param.element_size()
    
    for buffer in model.buffers():
        buffer_size += buffer.nelement() * buffer.element_size()
    
    total_size = param_size + buffer_size
    return total_size / (1024 * 1024)


In [ ]:
# Check initial memory usage
initial_memory = get_memory_usage()
print(f"🔍 Initial memory usage: {initial_memory:.1f} MB")


In [ ]:
# Load and inspect text encoder
print("📥 Loading Text Encoder (text_sonar_basic_encoder)...")

text_encoder = TextToEmbeddingModelPipeline(
    encoder="text_sonar_basic_encoder",
    tokenizer="text_sonar_basic_encoder"
)

encoder_memory = get_memory_usage()
print("✅ Text Encoder loaded")
print(f"Memory after encoder: {encoder_memory:.1f} MB")
print(f"Encoder memory usage: {encoder_memory - initial_memory:.1f} MB")

# Get encoder model details
encoder_model = text_encoder.model
enc_total, enc_trainable = count_parameters(encoder_model)
enc_size = get_model_size_mb(encoder_model)

print("\n📊 Text Encoder Stats:")
print(f"  - Total parameters: {enc_total:,}")
print(f"  - Trainable parameters: {enc_trainable:,}")
print(f"  - Model size: {enc_size:.1f} MB")


In [ ]:
# Load and inspect text decoder
print("📥 Loading Text Decoder (text_sonar_basic_decoder)...")

text_decoder = EmbeddingToTextModelPipeline(
    decoder="text_sonar_basic_decoder",
    tokenizer="text_sonar_basic_encoder"
)

decoder_memory = get_memory_usage()
print("✅ Text Decoder loaded")
print(f"Memory after decoder: {decoder_memory:.1f} MB")
print(f"Decoder memory usage: {decoder_memory - encoder_memory:.1f} MB")

# Get decoder model details
decoder_model = text_decoder.model
dec_total, dec_trainable = count_parameters(decoder_model)
dec_size = get_model_size_mb(decoder_model)

print("\n📊 Text Decoder Stats:")
print(f"  - Total parameters: {dec_total:,}")
print(f"  - Trainable parameters: {dec_trainable:,}")
print(f"  - Model size: {dec_size:.1f} MB")


In [ ]:
# Summary
total_memory = decoder_memory - initial_memory
total_params = enc_total + dec_total
total_size = enc_size + dec_size

print("📋 SUMMARY")
print("=" * 30)
print(f"🔢 Total Parameters: {total_params:,}")
print(f"📦 Total Model Size: {total_size:.1f} MB")
print(f"💾 Total Memory Usage: {total_memory:.1f} MB")
print("🎯 Embedding Dimension: 1024")


In [ ]:
# Test inference
print("🧪 Testing Inference...")

test_sentences = ["Hello world!", "This is a test sentence."]
embeddings = text_encoder.predict(test_sentences, source_lang="eng_Latn")
reconstructed = text_decoder.predict(embeddings, target_lang="eng_Latn")

inference_memory = get_memory_usage()
print("✅ Inference successful")
print(f"Input: {test_sentences}")
print(f"Reconstructed: {reconstructed}")
print(f"Peak memory during inference: {inference_memory:.1f} MB")
print(f"Inference overhead: {inference_memory - decoder_memory:.1f} MB")

print(f"\nEmbedding shape: {embeddings.shape}")
print(f"Embedding sample: {embeddings[0][:5]}...")  # First 5 dimensions
